In [ ]:
from nbdev import *

In [ ]:
#hide
import sys
sys.path.append("..")
import json
import numpy as np
import matplotlib.pyplot as plt
import gym
import tensorflow as tf
from dpct.individual import DHPCTIndividual
from dpct.evolver import DHPCTEvolver
from dpct.optimizer import DHPCTOptimizer

# DPCT Usage Examples

> Complete examples of using the DPCT library for deep perceptual control theory applications.

## 1. Basic Individual Usage

In [ ]:
# Create a simple individual for CartPole environment
individual = DHPCTIndividual(
    env_name="CartPole",
    gym_name="CartPole-v1",
    env_props={},
    levels=[
        {'units': 16},  # Level 1
        {'units': 8}    # Level 2
    ],
    activation_funcs=['relu', 'tanh'],
    weight_types=['glorot_uniform', 'glorot_uniform']
)

# Compile the individual
individual.compile()

# Display model summary
individual.model.summary()

In [ ]:
# Run the individual in the environment (with rendering disabled for notebook)
rewards = individual.run(episodes=5, render=False)
print(f"Rewards per episode: {rewards}")
print(f"Average reward: {np.mean(rewards)}")

In [ ]:
# Run with online learning enabled
rewards_with_learning = individual.run(episodes=5, render=False, online_learning=True, learning_rate=0.01)
print(f"Rewards with online learning: {rewards_with_learning}")
print(f"Average reward with learning: {np.mean(rewards_with_learning)}")

In [ ]:
# Save configuration to file
individual.save_config("cartpole_individual_config.json")

# Load from configuration
with open("cartpole_individual_config.json", "r") as f:
    config = json.load(f)
    
loaded_individual = DHPCTIndividual.from_config(config)
print("Successfully loaded individual from config!")

## 2. Evolution Example

In [ ]:
# Create a template individual for evolution
template = DHPCTIndividual(
    env_name="CartPole",
    gym_name="CartPole-v1",
    env_props={},
    levels=[
        {'units': 16},
        {'units': 8}
    ],
    activation_funcs=['relu', 'tanh'],
    weight_types=['glorot_uniform', 'glorot_uniform']
)

# Initialize the evolver with small population and generations for demonstration
evolver = DHPCTEvolver(
    population_size=5,   # Small size for demo
    gens=3,              # Few generations for demo
    cx_prob=0.7,
    mut_prob=0.2,
    elite_size=1,
    env_template=template
)

In [ ]:
# Setup evolution
evolver.setup_evolution(evaluation_episodes=2, mutation_rate=0.1)

In [ ]:
# Run evolution (uncomment to run, but it will take time)
# pop, log = evolver.run_evolution(verbose=True)

In [ ]:
# Save evolution results (commented out as it depends on evolution running)
'''
evolver.save_results(
    population_file="cartpole_population.json",
    logbook_file="cartpole_evolution_log.json",
    best_file="cartpole_best_evolved.json"
)
'''

In [ ]:
# Plot evolution progress (commented out as it depends on evolution running)
'''
fig = evolver.plot_evolution()
plt.show()
'''

## 3. Hyperparameter Optimization Example

In [ ]:
# Initialize the optimizer with small number of trials for demonstration
optimizer = DHPCTOptimizer(
    env_name="CartPole",
    gym_name="CartPole-v1",
    env_props={},
    n_trials=5,    # Small number for demo
    timeout=None,
    db_storage=None
)

In [ ]:
# Run optimization (commented out as it would take time)
'''
study = optimizer.run_optimization(evaluation_episodes=2, verbose=True)
'''

In [ ]:
# Get best individual and evaluate it (commented out as it depends on optimization)
'''
best_individual = optimizer.get_best_individual()
best_individual.compile()
print("Best Individual Model Summary:")
best_individual.model.summary()

# Evaluate best individual
rewards = best_individual.run(episodes=5, render=False)
print(f"Best individual rewards: {rewards}")
print(f"Average reward: {np.mean(rewards)}")
'''

In [ ]:
# Save optimization results (commented out as it depends on optimization running)
'''
optimizer.save_results(
    best_params_file="cartpole_best_params.json",
    best_individual_file="cartpole_best_optimized.json"
)
'''

## 4. Combined Example: Optimize, Evolve, and Use Online Learning

In [ ]:
# This is a conceptual example of how all components can work together
# Each step can be time-consuming, so the code is provided as a reference

'''
# Step 1: Optimize hyperparameters
optimizer = DHPCTOptimizer("CartPole", "CartPole-v1", n_trials=20)
optimizer.run_optimization(evaluation_episodes=3)
template = optimizer.get_best_individual()

# Step 2: Evolve population based on optimized template
evolver = DHPCTEvolver(population_size=20, gens=20, env_template=template)
evolver.run_evolution()
best_individual = evolver.best_individual

# Step 3: Fine-tune with online learning
best_individual.compile()
rewards = best_individual.run(episodes=20, online_learning=True, learning_rate=0.01)

# Display final results
print(f"Final performance: {np.mean(rewards)}")
'''

## 5. Using with Different Environments

In [ ]:
# Example with a different OpenAI Gym environment: MountainCar
# Note that this environment has a different observation and action space

'''
# Create individual for MountainCar
mountain_car = DHPCTIndividual(
    env_name="MountainCar",
    gym_name="MountainCar-v0",
    env_props={},
    levels=[
        {'units': 24},
        {'units': 12},
        {'units': 6}
    ],
    activation_funcs=['relu', 'tanh', 'tanh'],
    weight_types=['glorot_uniform', 'glorot_uniform', 'glorot_uniform']
)

# Compile and run
mountain_car.compile()
mountain_car.model.summary()
rewards = mountain_car.run(episodes=3, render=False)
print(f"MountainCar rewards: {rewards}")
'''

# DPCT Usage Examples

> Examples of using the Deep Perceptual Control Theory library.

In [ ]:
import sys
sys.path.append("..")
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import gym
import optuna

In [ ]:
# Import DPCT components
from dpct.individual import DHPCTIndividual
from dpct.evolver import DHPCTEvolver
from dpct.optimizer import DHPCTOptimizer

## 1. Creating and Running an Individual

In [ ]:
# Create an individual for the CartPole environment
individual = DHPCTIndividual(
    env_name="CartPole",
    gym_name="CartPole-v1",
    env_props={"render_mode": "rgb_array"},
    levels=[4, 3, 2],  # 4 inputs in lowest level, 3 in middle, 2 in highest
    activation_funcs={0: "linear", 1: "relu", 2: "tanh"},
    weight_types={"all": "float"}
)

In [ ]:
# Compile the individual (creates environment and builds model)
individual.compile()

In [ ]:
# Run the individual for 100 steps
reward = individual.run(steps=100, early_termination=True)
print(f"Reward: {reward}")

In [ ]:
# Run with online learning enabled
reward = individual.run(steps=100, train=True, early_termination=True)
print(f"Reward with training: {reward}")

## 2. Configuration Management

In [ ]:
# Get current configuration
config = individual.config()
print("Configuration:")
print(json.dumps(config, indent=2))

In [ ]:
# Save configuration to file
individual.save_config("individual_config.json")

In [ ]:
# Load configuration and create a new individual
with open("individual_config.json", "r") as f:
    loaded_config = json.load(f)

new_individual = DHPCTIndividual.from_config(loaded_config)
new_individual.compile()

## 3. Evolution

In [ ]:
# Define a fitness function
def fitness_function(individual):
    # For CartPole, we want to maximize the reward, so return negative reward
    # (since DEAP minimizes by default)
    return -individual.evaluate(nevals=3)

In [ ]:
# Create an evolver
evolver = DHPCTEvolver(
    pop_size=10,  # Small population for example
    generations=5,  # Few generations for example
    evolve_static_termination=True,
    unchanged_generations=3,
    run_best=True,
    save_arch_best=True
)

In [ ]:
# Setup evolution
evolver.setup_evolution(
    template_individual=individual,
    fitness_function=fitness_function,
    minimize=True  # We're minimizing negative reward
)

In [ ]:
# Run evolution (commented out for notebook example as it takes time)
# population, logbook, hof = evolver.run_evolution()

## 4. Hyperparameter Optimization

In [ ]:
# Define evolution parameters for optimization
evolution_params = {
    "pop_size": {"fixed": False, "type": "int", "min": 5, "max": 20, "step": 5},
    "generations": {"fixed": True, "value": 5},
    "evolve_static_termination": {"fixed": True, "value": True},
    "unchanged_generations": {"fixed": False, "type": "int", "min": 2, "max": 4},
    "minimize": {"fixed": True, "value": True}
}

In [ ]:
# Create an optimizer
optimizer = DHPCTOptimizer(
    evolution_params=evolution_params,
    n_trials=3,  # Few trials for example
    pruner=optuna.pruners.MedianPruner()
)

In [ ]:
# Define the objective
optimizer.define_objective(
    template_individual=individual,
    fitness_function=fitness_function,
    evaluation_budget=5  # Limit evaluation budget for example
)

In [ ]:
# Run optimization (commented out for notebook example as it takes time)
# study = optimizer.run_optimization()

In [ ]:
# Get best parameters (example output)
best_params = {
    "pop_size": 15,
    "generations": 5,
    "evolve_static_termination": True,
    "unchanged_generations": 3,
    "minimize": True
}
print("Best parameters:")
print(json.dumps(best_params, indent=2))

## 5. Online Learning

In [ ]:
# Create and compile an individual for online learning
online_learner = DHPCTIndividual(
    env_name="CartPole",
    gym_name="CartPole-v1",
    env_props={"render_mode": "rgb_array"},
    levels=[4, 3, 2]
)
online_learner.compile()

In [ ]:
# Run first without training to establish baseline
print("Running without training...")
rewards_without_training = []
for _ in range(5):
    reward = online_learner.run(steps=100, train=False, early_termination=True)
    rewards_without_training.append(reward)
    print(f"Reward: {reward}")

print(f"Average reward without training: {sum(rewards_without_training) / len(rewards_without_training)}")

In [ ]:
# Run with online learning
print("Running with training...")
rewards_with_training = []
for i in range(10):
    reward = online_learner.run(steps=100, train=True, early_termination=True)
    rewards_with_training.append(reward)
    print(f"Episode {i+1} reward: {reward}")

print(f"Average reward with training: {sum(rewards_with_training) / len(rewards_with_training)}")

In [ ]:
# Plot learning progress
plt.figure(figsize=(10, 6))
plt.plot(rewards_with_training, 'b-', label='With Online Learning')
plt.axhline(y=sum(rewards_without_training) / len(rewards_without_training), 
           color='r', linestyle='-', label='Without Training Baseline')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Online Learning Progress')
plt.legend()
plt.grid(True)
plt.show()